In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.api import VAR

# ============================================================
# TVP-VAR (Kalman Filter 기반, revised with output artifacts)
# ------------------------------------------------------------
# 입력: transformed / scaled data
# lag : BIC 선택
#
# 출력:
#   - tvpvar_beta.npy
#   - tvpvar_cov.npy
#   - tvpvar_selected_lag.txt
#   - tvpvar_effective_dates.csv
#   - tvpvar_var_names.csv
#   - tvpvar_diag_summary.csv
#
# 핵심 수정
# 1) P_t를 모든 방정식이 공유하지 않고 식별별로 분리
# 2) GFEVD 코드와 맞물리도록 유효 날짜 산출물 추가
# 3) 변수 순서 메타데이터 저장
# 4) TVP-VAR 시점별 진단 요약 저장
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

OUT_BETA = BASE_DIR / "tvpvar_beta.npy"
OUT_COV = BASE_DIR / "tvpvar_cov.npy"
OUT_LAG = BASE_DIR / "tvpvar_selected_lag.txt"
OUT_DATES = BASE_DIR / "tvpvar_effective_dates.csv"
OUT_VARNAMES = BASE_DIR / "tvpvar_var_names.csv"
OUT_DIAG = BASE_DIR / "tvpvar_diag_summary.csv"

# Kalman / covariance 설정
LAMBDA = 0.99
DELTA = 0.01
COV_ALPHA = 0.05
RIDGE_EPS = 1e-8
INIT_P_SCALE = 0.1
Q_SCALE = 1e-4

DATE_CANDIDATES = ["Date", "date", "DATE"]

VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

# -----------------------------
# Helpers
# -----------------------------
def find_date_col(df: pd.DataFrame):
    for c in DATE_CANDIDATES:
        if c in df.columns:
            return c
    return None


def run_select_order_safely(model, requested_maxlags: int = 10, min_candidate_lag: int = 1):
    last_error = None
    for maxlags in range(requested_maxlags, min_candidate_lag - 1, -1):
        try:
            sel = model.select_order(maxlags=maxlags)
            aic_list = sel.ics.get("aic", None)
            if aic_list is not None and len(aic_list) >= 2:
                return sel, maxlags
        except Exception as e:
            last_error = e
            continue

    raise RuntimeError(
        f"Unable to run VAR.select_order with lag >= {min_candidate_lag}. "
        f"Last error: {repr(last_error)}"
    )


def create_lagged_matrix(Y, p):
    T, k = Y.shape
    X = []

    for t in range(p, T):
        row = []
        for lag in range(1, p + 1):
            row.extend(Y[t - lag])
        X.append(row)

    return np.array(X)


# -----------------------------
# Load
# -----------------------------
raw_df = pd.read_csv(INPUT_FILE)

date_col = find_date_col(raw_df)

if date_col is not None:
    raw_df[date_col] = pd.to_datetime(raw_df[date_col], errors="coerce")
    raw_df = raw_df.sort_values(date_col).reset_index(drop=True)

missing_cols = [c for c in VAR_NAMES if c not in raw_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in input file: {missing_cols}")

work_df = raw_df.copy()

# 숫자형 강제 변환
for c in VAR_NAMES:
    work_df[c] = pd.to_numeric(work_df[c], errors="coerce")

keep_cols = VAR_NAMES.copy()
if date_col is not None:
    keep_cols = [date_col] + keep_cols

work_df = work_df[keep_cols].dropna().reset_index(drop=True)

if len(work_df) < 10:
    raise ValueError(f"Too few usable rows after cleaning: {len(work_df)}")

if date_col is not None:
    effective_source_dates = work_df[date_col].copy().reset_index(drop=True)
else:
    effective_source_dates = pd.Series(np.arange(len(work_df)), name="time_index")

df = work_df[VAR_NAMES].copy()

Y = df.values
T, k = Y.shape

print("Data shape:", Y.shape)
print("Using columns:", VAR_NAMES)

# -----------------------------
# Step 1. Lag selection (BIC)
# -----------------------------
var_model = VAR(df)
lag_order, used_maxlags = run_select_order_safely(var_model, requested_maxlags=10, min_candidate_lag=1)

p = lag_order.selected_orders["bic"]
if p is None or p < 1:
    p = 1

print("Selected lag (BIC):", p)
print("Used maxlags      :", used_maxlags)

with open(OUT_LAG, "w", encoding="utf-8") as f:
    f.write(str(p))

# -----------------------------
# Step 2. Prepare matrices
# -----------------------------
X = create_lagged_matrix(Y, p)     # (T_eff, k*p)
Y_trim = Y[p:]                     # (T_eff, k)
dates_trim = effective_source_dates.iloc[p:].reset_index(drop=True)

T_eff = Y_trim.shape[0]
k_beta = k * p

print("Lagged X shape :", X.shape)
print("Trimmed Y shape:", Y_trim.shape)
print("k_beta         :", k_beta)

if len(dates_trim) != T_eff:
    raise ValueError(f"dates_trim length({len(dates_trim)}) != T_eff({T_eff})")

# -----------------------------
# Step 3. Kalman Filter
# -----------------------------
beta_t = np.zeros((T_eff, k, k_beta))
cov_t = np.zeros((T_eff, k, k))

# equation별 state covariance
P_t = np.array([np.eye(k_beta) * INIT_P_SCALE for _ in range(k)])   # (k, k_beta, k_beta)

# observation covariance
R_diag = np.full(k, DELTA)

# state innovation covariance
Q = np.eye(k_beta) * Q_SCALE

# initial beta
beta_prev = np.zeros((k, k_beta))

# EWMA innovation covariance initializer
Sigma_ewma = np.cov(Y_trim.T)
Sigma_ewma = np.atleast_2d(Sigma_ewma)
Sigma_ewma = 0.5 * (Sigma_ewma + Sigma_ewma.T)
Sigma_ewma += np.eye(k) * RIDGE_EPS

print("Initial Sigma_ewma diag:", np.round(np.diag(Sigma_ewma), 6))

diag_rows = []

for t in range(T_eff):
    x_t = X[t]
    y_t = Y_trim[t]

    beta_new = np.zeros((k, k_beta))
    res_vec = np.zeros(k)

    for i in range(k):
        beta_prev_i = beta_prev[i].reshape(-1, 1)
        P_prev_i = P_t[i]

        # Prediction
        beta_pred_i = beta_prev_i
        P_pred_i = P_prev_i / LAMBDA + Q

        # Measurement
        H_i = x_t.reshape(1, -1)
        y_pred_i = float(H_i @ beta_pred_i)
        resid_i = float(y_t[i] - y_pred_i)

        S_i = float(H_i @ P_pred_i @ H_i.T + R_diag[i])
        if S_i <= 0:
            S_i = RIDGE_EPS

        K_i = (P_pred_i @ H_i.T) / S_i

        # Update
        beta_upd_i = beta_pred_i + K_i * resid_i
        P_upd_i = (np.eye(k_beta) - K_i @ H_i) @ P_pred_i
        P_upd_i = 0.5 * (P_upd_i + P_upd_i.T)
        P_upd_i += np.eye(k_beta) * RIDGE_EPS

        beta_new[i] = beta_upd_i.ravel()
        res_vec[i] = resid_i
        P_t[i] = P_upd_i

    beta_prev = beta_new
    beta_t[t] = beta_new

    # EWMA innovation covariance update
    outer_res = np.outer(res_vec, res_vec)
    Sigma_ewma = (1.0 - COV_ALPHA) * Sigma_ewma + COV_ALPHA * outer_res
    Sigma_ewma = 0.5 * (Sigma_ewma + Sigma_ewma.T)
    Sigma_ewma += np.eye(k) * RIDGE_EPS

    cov_t[t] = Sigma_ewma

    mean_abs_beta = float(np.mean(np.abs(beta_new)))
    mean_cov_diag = float(np.mean(np.diag(Sigma_ewma)))
    mean_p_diag = float(np.mean([np.mean(np.diag(P_t[i])) for i in range(k)]))
    min_cov_eig = float(np.min(np.linalg.eigvalsh(Sigma_ewma)))
    min_p_eig = float(np.min([np.min(np.linalg.eigvalsh(P_t[i])) for i in range(k)]))

    diag_rows.append({
        "t": t,
        "Date": dates_trim.iloc[t] if date_col is not None else int(dates_trim.iloc[t]),
        "mean_abs_beta": mean_abs_beta,
        "mean_diag_cov": mean_cov_diag,
        "mean_diag_P": mean_p_diag,
        "min_cov_eig": min_cov_eig,
        "min_P_eig": min_p_eig,
    })

    if t % 200 == 0 or t == T_eff - 1:
        print(
            f"t={t}/{T_eff} | "
            f"mean|beta|={mean_abs_beta:.6f} | "
            f"mean diag(cov)={mean_cov_diag:.6f} | "
            f"mean diag(P)={mean_p_diag:.6f}"
        )

# -----------------------------
# Save outputs
# -----------------------------
np.save(OUT_BETA, beta_t)
np.save(OUT_COV, cov_t)

# 1) effective dates
df_dates = pd.DataFrame({
    "t": np.arange(T_eff),
    "Date": dates_trim.astype(str).values if date_col is not None else dates_trim.values
})
df_dates.to_csv(OUT_DATES, index=False, encoding="utf-8-sig")

# 2) variable names / order
df_varnames = pd.DataFrame({
    "idx": np.arange(k),
    "var_name": VAR_NAMES
})
df_varnames.to_csv(OUT_VARNAMES, index=False, encoding="utf-8-sig")

# 3) diagnostic summary
df_diag = pd.DataFrame(diag_rows)
df_diag.to_csv(OUT_DIAG, index=False, encoding="utf-8-sig")

print("\nSaved:")
print(" -", OUT_BETA)
print(" -", OUT_COV)
print(" -", OUT_LAG)
print(" -", OUT_DATES)
print(" -", OUT_VARNAMES)
print(" -", OUT_DIAG)

# -----------------------------
# Sanity checks
# -----------------------------
print("\n[Sanity Check]")
print("beta_t shape:", beta_t.shape)
print("cov_t shape :", cov_t.shape)
print("P_t shape   :", P_t.shape)
print("dates len   :", len(df_dates))
print("var count   :", len(df_varnames))
print("last beta mean abs:", np.mean(np.abs(beta_t[-1])))
print("last cov diag     :", np.round(np.diag(cov_t[-1]), 6))
print("last cov eig min  :", np.min(np.linalg.eigvalsh(cov_t[-1])))

min_eigs_P = [np.min(np.linalg.eigvalsh(P_t[i])) for i in range(k)]
print("last P_t min eig by eq:", np.round(min_eigs_P, 10))

Data shape: (1036, 7)
Using columns: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_GOLD', 'dlog_SILVER', 'dlog_DXY', 'd_UST10Y', 'd_VIX']
Selected lag (BIC): 1
Used maxlags      : 10
Lagged X shape : (1035, 7)
Trimmed Y shape: (1035, 7)
k_beta         : 7
Initial Sigma_ewma diag: [1.001773 1.001789 0.999179 0.997618 1.000913 1.001582 1.001664]
t=0/1035 | mean|beta|=0.041033 | mean diag(cov)=0.964003 | mean diag(P)=0.086817
t=200/1035 | mean|beta|=0.198886 | mean diag(cov)=0.776013 | mean diag(P)=0.002124


C:\Users\User\AppData\Local\Temp\ipykernel_12820\1441542871.py:223: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y_pred_i = float(H_i @ beta_pred_i)
C:\Users\User\AppData\Local\Temp\ipykernel_12820\1441542871.py:226: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  S_i = float(H_i @ P_pred_i @ H_i.T + R_diag[i])


t=400/1035 | mean|beta|=0.250973 | mean diag(cov)=2.590457 | mean diag(P)=0.001661
t=600/1035 | mean|beta|=0.185939 | mean diag(cov)=1.155190 | mean diag(P)=0.002379
t=800/1035 | mean|beta|=0.173196 | mean diag(cov)=1.316053 | mean diag(P)=0.001938
t=1000/1035 | mean|beta|=0.260499 | mean diag(cov)=1.511705 | mean diag(P)=0.001612
t=1034/1035 | mean|beta|=0.225605 | mean diag(cov)=1.694112 | mean diag(P)=0.001815

Saved:
 - tvpvar_beta.npy
 - tvpvar_cov.npy
 - tvpvar_selected_lag.txt
 - tvpvar_effective_dates.csv
 - tvpvar_var_names.csv
 - tvpvar_diag_summary.csv

[Sanity Check]
beta_t shape: (1035, 7, 7)
cov_t shape : (1035, 7, 7)
P_t shape   : (7, 7, 7)
dates len   : 1035
var count   : 7
last beta mean abs: 0.22560492013948236
last cov diag     : [1.369312 2.691009 2.576623 3.745621 0.548809 0.364522 0.562889]
last cov eig min  : 0.24468810578790398
last P_t min eig by eq: [0.00026152 0.00026152 0.00026152 0.00026152 0.00026152 0.00026152
 0.00026152]
